# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [7]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

IMG_SIZE = 384 
BATCH_SIZE = 8

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED SPLIT
# ==========================================
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

train_df, valid_df = train_test_split(
    df_all, test_size=0.2, stratify=df_all['label'], random_state=42
)

# ==========================================
# 3. PIPELINE AUGMENTASI (ALBUMENTATIONS)
# ==========================================
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, p=0.5),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 5. INISIALISASI DATALOADER
# ==========================================
train_dataset = SpoofingDataset(train_df, transforms=train_transforms)
valid_dataset = SpoofingDataset(valid_df, transforms=valid_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Prapemrosesan Selesai!")
print(f"Total Data Latih    : {len(train_dataset)} gambar")
print(f"Total Data Validasi : {len(valid_dataset)} gambar")
print(f"Kamus Label         : {id2label}")

Prapemrosesan Selesai!
Total Data Latih    : 1321 gambar
Total Data Validasi : 331 gambar
Kamus Label         : {0: 'fake_mannequin', 1: 'fake_mask', 2: 'fake_printed', 3: 'fake_screen', 4: 'fake_unknown', 5: 'realperson'}


# Membangun Arsitektur Swin dan CNN

In [8]:
import torch
import torch.nn as nn
import timm

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. INISIALISASI DUA MODEL (SWIN & CNN)
# ==========================================
NUM_CLASSES = 6

# --- MODEL 1: SWIN TRANSFORMER ---
MODEL_SWIN_NAME = 'swin_base_patch4_window12_384.ms_in22k'
print(f"Mengunduh dan membangun model 1: {MODEL_SWIN_NAME}...")
model_swin = timm.create_model(
    MODEL_SWIN_NAME, 
    pretrained=True, 
    num_classes=NUM_CLASSES,
    drop_rate=0.3,       
    attn_drop_rate=0.2   
)
model_swin = model_swin.to(device)

# --- MODEL 2: EFFICIENTNET V2 ---
MODEL_CNN_NAME = 'tf_efficientnetv2_m.in21k_ft_in1k'
print(f"Mengunduh dan membangun model 2: {MODEL_CNN_NAME}...")
model_cnn = timm.create_model(
    MODEL_CNN_NAME, 
    pretrained=True, 
    num_classes=NUM_CLASSES,
    drop_rate=0.3,       
    drop_path_rate=0.2   
)
model_cnn = model_cnn.to(device)

# ==========================================
# 3. UJI COBA KEDUA MODEL (SANITY CHECK)
# ==========================================
dummy_input = torch.randn(4, 3, 384, 384).to(device)

with torch.no_grad():
    dummy_out_swin = model_swin(dummy_input)
    dummy_out_cnn = model_cnn(dummy_input)

print("\nKedua model berhasil dibangun dan dipindahkan ke GPU!")
print(f"Bentuk input                : {dummy_input.shape}")
print(f"Bentuk output Swin          : {dummy_out_swin.shape}")
print(f"Bentuk output EfficientNet  : {dummy_out_cnn.shape}")

Menggunakan perangkat: cuda
Mengunduh dan membangun model 1: swin_base_patch4_window12_384.ms_in22k...
Mengunduh dan membangun model 2: tf_efficientnetv2_m.in21k_ft_in1k...


model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]


Kedua model berhasil dibangun dan dipindahkan ke GPU!
Bentuk input                : torch.Size([4, 3, 384, 384])
Bentuk output Swin          : torch.Size([4, 6])
Bentuk output EfficientNet  : torch.Size([4, 6])


# Strategi Pelatihan (Warm-up & Loss)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np
import gc

# ==========================================
# 1. DEFINISI FOCAL LOSS & FUNGSI PELATIHAN
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

# Fungsi modular untuk melatih model apa saja
def train_model(model, model_name, save_path, epochs=30, lr=1e-5, patience=10, accumulation_steps=4):
    print(f"\n{'='*50}")
    print(f"Memulai Pelatihan: {model_name}")
    print(f"{'='*50}")
    
    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler()

    best_val_f1 = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        # --- FASE TRAINING ---
        model.train()
        train_loss = 0.0
        optimizer.zero_grad() 
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train {model_name}]")
        for batch_idx, (images, labels) in enumerate(train_bar):
            images, labels = images.to(device), labels.to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(images) 
                loss = criterion(outputs, labels) 
                loss = loss / accumulation_steps 
            
            scaler.scale(loss).backward() 
            
            if ((batch_idx + 1) % accumulation_steps == 0) or (batch_idx + 1 == len(train_loader)):
                scaler.step(optimizer) 
                scaler.update()
                optimizer.zero_grad() 
            
            train_loss += loss.item() * accumulation_steps * images.size(0)
            train_bar.set_postfix(loss=(loss.item() * accumulation_steps))
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # --- FASE VALIDASI ---
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid {model_name}]")
        with torch.no_grad(): 
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\nHasil Epoch {epoch+1}:")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
        
        scheduler.step()
        
        # --- SIMPAN & EARLY STOPPING ---
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), save_path)
            print(f"Model membaik! Disimpan ke '{save_path}'")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"Model tidak membaik. Kesabaran: {patience_counter}/{patience}")
            
        if patience_counter >= patience:
            print(f"\nEarly Stopping dipicu untuk {model_name}!")
            break

    print(f"\nPelatihan {model_name} Selesai! Skor Macro F1 Validasi Terbaik: {best_val_f1:.4f}")
    return best_val_f1

# ==========================================
# 2. EKSEKUSI PELATIHAN DUA MODEL
# ==========================================

# 1. Melatih Swin Transformer
swin_best_f1 = train_model(
    model=model_swin, 
    model_name="Swin Transformer", 
    save_path="best_swin_model.pth"
)

# PEMBERSIHAN MEMORI GPU (Sangat Penting)
print("\nMembersihkan memori GPU sebelum melatih model kedua...")
del model_swin
torch.cuda.empty_cache()
gc.collect()

# 2. Melatih EfficientNetV2
cnn_best_f1 = train_model(
    model=model_cnn, 
    model_name="EfficientNet V2", 
    save_path="best_cnn_model.pth",
    lr=2e-5 # Model CNN biasanya butuh learning rate sedikit lebih besar dari Transformer
)

print(f"\n{'='*50}")
print("REKAPITULASI PELATIHAN")
print(f"Swin Transformer F1  : {swin_best_f1:.4f}")
print(f"EfficientNetV2 F1    : {cnn_best_f1:.4f}")
print(f"{'='*50}")


Memulai Pelatihan: Swin Transformer


Epoch 1/30 [Valid Swin Transformer]: 100%|██████████| 42/42 [00:06<00:00,  6.21it/s]



Hasil Epoch 1:
Train Loss: 1.0086 | Val Loss: 0.6331 | Val Macro F1: 0.5766
Model membaik! Disimpan ke 'best_swin_model.pth'


Epoch 2/30 [Valid Swin Transformer]: 100%|██████████| 42/42 [00:06<00:00,  6.07it/s]



Hasil Epoch 2:
Train Loss: 0.5785 | Val Loss: 0.4721 | Val Macro F1: 0.7013
Model membaik! Disimpan ke 'best_swin_model.pth'


...


Hasil Epoch 28:
Train Loss: 0.5546 | Val Loss: 0.7400 | Val Macro F1: 0.7198
Model membaik! Disimpan ke 'best_cnn_model.pth'


Epoch 29/30 [Valid EfficientNet V2]: 100%|██████████| 42/42 [00:06<00:00,  6.84it/s]



Hasil Epoch 29:
Train Loss: 0.5334 | Val Loss: 0.7494 | Val Macro F1: 0.7164
Model tidak membaik. Kesabaran: 1/10


Epoch 30/30 [Valid EfficientNet V2]: 100%|██████████| 42/42 [00:05<00:00,  7.29it/s]


Hasil Epoch 30:
Train Loss: 0.5275 | Val Loss: 0.7313 | Val Macro F1: 0.6963
Model tidak membaik. Kesabaran: 2/10

Pelatihan EfficientNet V2 Selesai! Skor Macro F1 Validasi Terbaik: 0.7198

REKAPITULASI PELATIHAN
Swin Transformer F1  : 0.8168
EfficientNetV2 F1    : 0.7198


# Kalibrasi Probabilitas (Thresholding)

In [10]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize
import warnings
import timm
import gc

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6

# ==========================================
# 1. EKSTRAKSI PROBABILITAS: SWIN TRANSFORMER
# ==========================================
print("=== EKSTRAKSI PROBABILITAS: SWIN TRANSFORMER ===")
# Membangun ulang arsitektur (pretrained=False karena kita akan memuat bobot lokal)
model_swin = timm.create_model('swin_base_patch4_window12_384.ms_in22k', pretrained=False, num_classes=NUM_CLASSES)
model_swin.load_state_dict(torch.load('best_swin_model.pth', map_location=device))
model_swin.to(device)
model_swin.eval()

swin_probs_list = []
all_labels = []

with torch.no_grad():
    for images, labels in valid_loader: 
        images = images.to(device)
        outputs = model_swin(images)
        probs = F.softmax(outputs, dim=1) 
        swin_probs_list.extend(probs.cpu().numpy())
        
        # Simpan label asli cukup satu kali saja
        if len(all_labels) < len(valid_loader.dataset):
            all_labels.extend(labels.numpy())

# Membersihkan VRAM GPU agar tidak Out of Memory
del model_swin
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 2. EKSTRAKSI PROBABILITAS: EFFICIENTNET V2
# ==========================================
print("=== EKSTRAKSI PROBABILITAS: EFFICIENTNET V2 ===")
model_cnn = timm.create_model('tf_efficientnetv2_m.in21k_ft_in1k', pretrained=False, num_classes=NUM_CLASSES)
model_cnn.load_state_dict(torch.load('best_cnn_model.pth', map_location=device))
model_cnn.to(device)
model_cnn.eval()

cnn_probs_list = []

with torch.no_grad():
    for images, labels in valid_loader: 
        images = images.to(device)
        outputs = model_cnn(images)
        probs = F.softmax(outputs, dim=1) 
        cnn_probs_list.extend(probs.cpu().numpy())

# Membersihkan VRAM GPU
del model_cnn
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 3. PENGGABUNGAN (ENSEMBLE) & EVALUASI AWAL
# ==========================================
swin_probs = np.array(swin_probs_list)
cnn_probs = np.array(cnn_probs_list)
all_labels = np.array(all_labels)

# Merata-ratakan prediksi kedua model (Bobot 50:50)
all_probs = (swin_probs + cnn_probs) / 2.0

preds_before = np.argmax(all_probs, axis=1)
f1_before = f1_score(all_labels, preds_before, average='macro')

print("\n--- SEBELUM KALIBRASI (HASIL ENSEMBLE) ---")
print(f"Macro F1-Score: {f1_before:.5f}")
print(classification_report(all_labels, preds_before, target_names=class_names, digits=4))

# ==========================================
# 4. PROSES OPTIMASI THRESHOLD
# ==========================================
def optimize_f1(weights):
    weighted_probs = all_probs * weights
    calibrated_preds = np.argmax(weighted_probs, axis=1)
    return -1.0 * f1_score(all_labels, calibrated_preds, average='macro')

initial_weights = [1.0] * 6 
bounds = [(0.1, 3.0)] * 6 

print("\nMemulai optimasi kalibrasi probabilitas (Metode: Powell)...")
result = minimize(optimize_f1, initial_weights, method='Powell', bounds=bounds)
best_weights = result.x

# ==========================================
# 5. EVALUASI SETELAH KALIBRASI
# ==========================================
weighted_probs_after = all_probs * best_weights
preds_after = np.argmax(weighted_probs_after, axis=1)
f1_after = f1_score(all_labels, preds_after, average='macro')

print("\n--- SETELAH KALIBRASI ---")
print(f"Macro F1-Score: {f1_after:.5f}")
print(f"Peningkatan   : {f1_after - f1_before:.5f}")
print(classification_report(all_labels, preds_after, target_names=class_names, digits=4))

print("\nSimpan array bobot ini untuk dipakai di Test Set (Tahap 5):")
print(f"best_weights = {list(best_weights)}")

=== EKSTRAKSI PROBABILITAS: SWIN TRANSFORMER ===
=== EKSTRAKSI PROBABILITAS: EFFICIENTNET V2 ===

--- SEBELUM KALIBRASI (HASIL ENSEMBLE) ---
Macro F1-Score: 0.79723
                precision    recall  f1-score   support

fake_mannequin     0.8163    0.8889    0.8511        45
     fake_mask     0.8148    0.7857    0.8000        56
  fake_printed     0.6190    0.5652    0.5909        23
   fake_screen     0.8864    0.8478    0.8667        46
  fake_unknown     0.8254    0.7647    0.7939        68
    realperson     0.8500    0.9140    0.8808        93

      accuracy                         0.8248       331
     macro avg     0.8020    0.7944    0.7972       331
  weighted avg     0.8234    0.8248    0.8231       331


Memulai optimasi kalibrasi probabilitas (Metode: Powell)...

--- SETELAH KALIBRASI ---
Macro F1-Score: 0.80982
Peningkatan   : 0.01259
                precision    recall  f1-score   support

fake_mannequin     0.7925    0.9333    0.8571        45
     fake_mask     0.78

# Prediksi TTA & Pengumpulan (Submission)

In [11]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings
import timm
import gc

warnings.filterwarnings("ignore")

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

IMG_SIZE = 384
BATCH_SIZE = 8 

test_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        image = None
        
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6

# PENTING: Pastikan Anda memperbarui array ini dengan hasil output dari Tahap 4 TERBARU!
best_weights = np.array([
    1.2917031201343783, 
    1.1086726156966198, 
    0.7846563764554524, 
    0.7846559645046248, 
    1.0655135128696007, 
    0.7068118913219787
]) 

print(f"Memulai prediksi {len(sub_df)} data dengan TTA (Ensemble Mode)...")

# ==========================================
# 3. PREDIKSI MODEL 1: SWIN TRANSFORMER
# ==========================================
print("\n--- Memuat & Memprediksi dengan Swin Transformer ---")
model_swin = timm.create_model('swin_base_patch4_window12_384.ms_in22k', pretrained=False, num_classes=NUM_CLASSES)
model_swin.load_state_dict(torch.load('best_swin_model.pth', map_location=device))
model_swin.to(device)
model_swin.eval()

swin_preds_all = []

with torch.no_grad():
    for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc="Swin TTA"):
        imgs_orig = imgs_orig.to(device)
        imgs_flip = imgs_flip.to(device)

        out_orig = model_swin(imgs_orig)
        prob_orig = F.softmax(out_orig, dim=1)

        out_flip = model_swin(imgs_flip)
        prob_flip = F.softmax(out_flip, dim=1)

        prob_avg = (prob_orig + prob_flip) / 2.0
        swin_preds_all.extend(prob_avg.cpu().numpy())

# Bersihkan memori secara total agar aman untuk model berikutnya
del model_swin
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 4. PREDIKSI MODEL 2: EFFICIENTNET V2
# ==========================================
print("\n--- Memuat & Memprediksi dengan EfficientNet V2 ---")
model_cnn = timm.create_model('tf_efficientnetv2_m.in21k_ft_in1k', pretrained=False, num_classes=NUM_CLASSES)
model_cnn.load_state_dict(torch.load('best_cnn_model.pth', map_location=device))
model_cnn.to(device)
model_cnn.eval()

cnn_preds_all = []

with torch.no_grad():
    for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc="CNN TTA"):
        imgs_orig = imgs_orig.to(device)
        imgs_flip = imgs_flip.to(device)

        out_orig = model_cnn(imgs_orig)
        prob_orig = F.softmax(out_orig, dim=1)

        out_flip = model_cnn(imgs_flip)
        prob_flip = F.softmax(out_flip, dim=1)

        prob_avg = (prob_orig + prob_flip) / 2.0
        cnn_preds_all.extend(prob_avg.cpu().numpy())

# Bersihkan memori
del model_cnn
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 5. PENGGABUNGAN (ENSEMBLE) & PENYIMPANAN
# ==========================================
print("\n--- Menggabungkan Prediksi & Menyusun Submission ---")
swin_preds_all = np.array(swin_preds_all)
cnn_preds_all = np.array(cnn_preds_all)

# Rata-rata probabilitas dari kedua model
final_ensemble_probs = (swin_preds_all + cnn_preds_all) / 2.0

# Terapkan bobot kalibrasi (Powell) pada hasil gabungan
weighted_probs = final_ensemble_probs * best_weights
final_preds = np.argmax(weighted_probs, axis=1)

# Simpan ke CSV
final_labels = [id2label[pred] for pred in final_preds]
sub_df['label'] = final_labels
sub_df.to_csv('submission.csv', index=False)

print("Selesai! File 'submission.csv' berhasil dibuat dengan aman.")
print(sub_df.head())

Memulai prediksi 404 data dengan TTA (Ensemble Mode)...

--- Memuat & Memprediksi dengan Swin Transformer ---


Swin TTA: 100%|██████████| 51/51 [00:35<00:00,  1.43it/s]



--- Memuat & Memprediksi dengan EfficientNet V2 ---


CNN TTA: 100%|██████████| 51/51 [00:14<00:00,  3.57it/s]



--- Menggabungkan Prediksi & Menyusun Submission ---
Selesai! File 'submission.csv' berhasil dibuat dengan aman.
         id           label
0  test_001     fake_screen
1  test_002  fake_mannequin
2  test_003      realperson
3  test_004      realperson
4  test_005    fake_printed


In [12]:
#0,69286